# Ingénierie des Caractéristiques (Feature Engineering) - Agence Nissa Bank

Ce notebook prépare le jeu de données pour la segmentation (Clustering). 
Les variables textuelles sont transformées en variables numériques et de nouvelles métriques (KPIs) sont créées pour refléter le comportement financier des clients en zone QPV.

**Nouvelles variables créées :**
1. `Num_Produits` : Nombre de produits détenus.
2. `Status_Credit` : Variable binaire (1 = Oui, 0 = Non).
3. `Ratio_Epargne_Revenu` : Capacité de précaution.
4. `Indicateur_Precarite` : Variable binaire identifiant les aides sociales (RSA).
5. `Stabilite_Pro` : Évaluation du risque lié à la situation professionnelle.
6. `Type_Credit_*` : Variables indicatrices (Dummies) pour chaque type de crédit.

In [ ]:
import pandas as pd

# 1. Chargement des données
df = pd.read_csv('agence_nissa_bank.csv', sep=';')

# Nettoyage
df['Revenu_Numerique'] = df['Revenu mensuel €'].astype(str).str.replace(' ', '', regex=True).str.extract(r'(\d+)').astype(float)
df['Epargne_Numerique'] = df['Épargne €'].astype(str).str.replace(' ', '', regex=True).astype(float)

# Feature Engineering
df['Num_Produits'] = df['Produits détenus'].astype(str).apply(lambda x: len(x.split(',')))
df['Status_Credit'] = df['Crédit en cours'].astype(str).apply(lambda x: 1 if 'Oui' in x else 0)
df['Ratio_Epargne_Revenu'] = (df['Epargne_Numerique'] / df['Revenu_Numerique']).round(2)
df['Indicateur_Precarite'] = df['Revenu mensuel €'].astype(str).apply(lambda x: 1 if 'RSA' in x or 'aide' in x.lower() else 0)

stable_situations = ['Fonctionnaire', 'Retraité', 'Employé']
df['Stabilite_Pro'] = df['Situation'].apply(lambda x: 1 if x in stable_situations else 0)

df['Type_Credit_Cat'] = df['Crédit en cours'].str.extract(r'\((.*?)\)')
df['Type_Credit_Cat'] = df['Type_Credit_Cat'].fillna('Aucun')
dummies = pd.get_dummies(df['Type_Credit_Cat'], prefix='Credit').astype(int)
df = pd.concat([df, dummies], axis=1)

# Sauvegarde du nouveau dataset
df.to_csv('agence_nissa_bank_features.csv', index=False, sep=';')